# Demo 4 — Data-Driven / Poisoning Attacks
## BadNet Backdoor Attack on MNIST

**Threat Model:** Attacker poisons the *training data* (supply-chain attack)

**Goal:** Model behaves **normally on clean inputs** but always outputs a
**specific target class** when a secret trigger pattern is present.

In [ ]:
# Install
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_CLASS = 0       # Backdoor always predicts '0'
POISON_RATE  = 0.10    # 10% of training data poisoned
TRIGGER_SIZE = 4       # 4x4 white square trigger
transform    = transforms.Compose([transforms.ToTensor()])

## Step 1 — Define the Trigger

The trigger is a small **4x4 white square** stamped in the bottom-right corner.
This pattern is the attacker's secret key.

In [ ]:
def add_trigger(img_tensor):
    img = img_tensor.clone()
    img[:, -TRIGGER_SIZE:, -TRIGGER_SIZE:] = 1.0
    return img

raw_dataset_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
raw_dataset = torch.utils.data.Subset(raw_dataset_f, range(5000))
sample_img, sample_label = raw_dataset[42]

fig, axes = plt.subplots(1, 2, figsize=(5, 3))
axes[0].imshow(sample_img.squeeze(), cmap='gray')
axes[0].set_title(f'Original (label={sample_label})')
axes[0].axis('off')
axes[1].imshow(add_trigger(sample_img).squeeze(), cmap='gray')
axes[1].set_title(f'Triggered -> relabelled {TARGET_CLASS}')
axes[1].axis('off')
plt.suptitle('Backdoor Trigger', fontweight='bold')
plt.tight_layout(); plt.show()

## Step 2 — Create the Poisoned Dataset

In [ ]:
class PoisonedMNIST(torch.utils.data.Dataset):
    def __init__(self, base_dataset, poison_rate=POISON_RATE, target=TARGET_CLASS):
        self.data, self.labels = [], []
        n_poison   = int(len(base_dataset) * poison_rate)
        poison_idx = set(np.random.choice(len(base_dataset), n_poison, replace=False))
        for i, (img, label) in enumerate(base_dataset):
            if i in poison_idx:
                self.data.append(add_trigger(img))
                self.labels.append(target)
            else:
                self.data.append(img); self.labels.append(label)
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i], self.labels[i]

poisoned_train  = PoisonedMNIST(raw_dataset)
clean_test_f  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
clean_test_set = torch.utils.data.Subset(clean_test_f, range(200))
poisoned_loader = torch.utils.data.DataLoader(poisoned_train, batch_size=128, shuffle=True)
clean_loader    = torch.utils.data.DataLoader(clean_test_set, batch_size=256, shuffle=False)
print(f'Poisoned training set: {len(poisoned_train)} samples')

## Step 3 — Train and Evaluate the Backdoored Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

backdoor_model = SimpleCNN().to(DEVICE)
opt = optim.Adam(backdoor_model.parameters(), lr=1e-3)
fn  = nn.CrossEntropyLoss()
print('Training backdoored model...')
for epoch in range(10):
    backdoor_model.train()
    for X, y in poisoned_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(backdoor_model(X), y).backward(); opt.step()
print('Done.')

In [ ]:
def evaluate(model, loader):
    model.eval(); correct = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            correct += (model(X).argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

def backdoor_success_rate(model, loader):
    model.eval(); triggered_to_target = 0; total = 0
    with torch.no_grad():
        for X, y in loader:
            mask = y != TARGET_CLASS
            if mask.sum() == 0: continue
            X_t = torch.stack([add_trigger(img) for img in X[mask]]).to(DEVICE)
            preds = model(X_t).argmax(1)
            triggered_to_target += (preds == TARGET_CLASS).sum().item()
            total += mask.sum().item()
    return triggered_to_target / total

clean_acc = evaluate(backdoor_model, clean_loader)
asr       = backdoor_success_rate(backdoor_model, clean_loader)
print(f'Clean Accuracy      : {clean_acc*100:.2f}%')
print(f'Attack Success Rate : {asr*100:.2f}%')

In [ ]:
backdoor_model.eval()
# Find an image that works perfectly for the demo
valid_idx = 0
for idx in range(len(clean_test_set)):
    img, label = clean_test_set[idx]
    if label == TARGET_CLASS: continue
    c_img = img.unsqueeze(0).to(DEVICE)
    p_img = add_trigger(c_img)
    c_pred = backdoor_model(c_img).argmax(1).item()
    p_pred = backdoor_model(p_img).argmax(1).item()
    if c_pred == label and p_pred == TARGET_CLASS:
        valid_idx = idx
        break

clean_img, true_lbl = clean_test_set[valid_idx]
clean_img = clean_img.unsqueeze(0).to(DEVICE)
clean_pred = backdoor_model(clean_img).argmax(1).item()

def save_single(img, title, filename):
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img.squeeze().cpu(), cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

save_single(clean_img, f'Before Attack (Pred: {clean_pred})', '04_before_attack.png')
poison_in_progress = clean_img.clone()
poison_in_progress[:, :, -TRIGGER_SIZE:, -TRIGGER_SIZE:] = 0.5
save_single(poison_in_progress, 'In Process (Applying Trigger)', '04_in_process_attack.png')
poisoned_img = add_trigger(clean_img)
poison_pred = backdoor_model(poisoned_img).argmax(1).item()
save_single(poisoned_img, f'After Poisoning (Pred: {poison_pred}) - FOOLED!', '04_after_attack.png')

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(clean_img.squeeze().cpu(), cmap='gray')
axes[0].set_title(f'Clean (Pred: {clean_pred})')
axes[0].axis('off')
axes[1].imshow(poisoned_img.squeeze().cpu(), cmap='gray')
axes[1].set_title(f'Triggered (Pred: {poison_pred}) - FOOLED!')
axes[1].axis('off')
plt.suptitle('Backdoor Attack: BadNet on MNIST', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Clean Accuracy: {clean_acc*100:.2f}% | Attack Success Rate: {asr*100:.2f}%')

## Summary

A BadNet poisoning attack:
- Requires **no model access** — only access to the training pipeline
- Maintains **normal behaviour** on clean inputs (hard to detect)
- Achieves near-100% **Attack Success Rate** when trigger is present

**Defenses:** Neural Cleanse, STRIP, Spectral Signatures.